<a href="https://colab.research.google.com/github/ValentinaEmili/Ethnicity-recognition/blob/main/codebook/VQGAN/VQGAN_TT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# VQ-GAN architecture inspired to https://github.com/compvis/taming-transformers for high resolution image synthesis

In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import matplotlib.pyplot as plt
import time

In [ ]:
def nonlinearity(x):
  return x * torch.sigmoid(x)


def Normalize(in_channels):
  return torch.nn.GroupNorm(num_groups=32, num_channels=in_channels, eps=1e-6, affine=True)

class Upsample(nn.Module):
    def __init__(self, in_channels, with_conv=True):
        super().__init__()
        self.with_conv = with_conv
        if with_conv:
          self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)

    def forward(self,x):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if self.with_conv:
          x = self.conv(x)
        return x

class Downsample(nn.Module):
    def __init__(self, in_channels, with_conv=True):
        super().__init__()
        self.with_conv = with_conv
        if with_conv:
          self.conv = torch.nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2, padding=0)

    def forward(self, x):
        if self.with_conv:
          x = F.pad(x, (0, 1, 0, 1), mode="constant", value=0)
          x = self.conv(x)
        else:
          x = F.avg_pool2d(x, kernel_size=2, stride=2)
        return x

class ResnetBlock(nn.Module):
    def __init__(self, in_channels, out_channels=None, dropout=0.0):
        super().__init__()
        out_channels = in_channels if out_channels is None else out_channels

        self.norm1 = Normalize(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)

        self.norm2 = Normalize(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)

        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.shortcut = nn.Identity()



    def forward(self,x):
        h = self.norm1(x)
        h = nonlinearity(h)
        h = self.conv1(h)

        h = self.norm2(h)
        h = nonlinearity(h)
        h = self.dropout(h)
        h = self.conv2(h)

        return self.shortcut(x) + h

class AttnBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.norm = Normalize(channels)

        self.q = nn.Conv2d(channels, channels, kernel_size=1)

        self.k = nn.Conv2d(channels, channels, kernel_size=1)

        self.v = nn.Conv2d(channels, channels, kernel_size=1)

        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)


    def forward(self,x):
        h_ = self.norm(x)

        q = self.q(h_)
        k = self.k(h_)
        v = self.v(h_)


        b,c,h,w = q.shape

        q = q.reshape(b,c,h*w)
        q = q.permute(0,2,1)

        k = k.reshape(b,c,h*w)

        attention = torch.bmm(q,k)
        attention = attention * (c**-0.5)
        attention = F.softmax(attention, dim=2)

        v = v.reshape(b,c,h*w)

        attention = attention.permute(0,2,1)

        out = torch.bmm(v,attention)
        out = out.reshape(b,c,h,w)

        out = self.proj_out(out)

        return x + out

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels=3, ch=128, ch_mult=(1, 1, 2, 2, 4), num_res_blocks=2, attn_resolutions=(16,), resolution=256, z_channels=256, dropout=0.0, resamp_with_conv=True):
        super().__init__()
        self.ch = ch
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution

        self.conv_in = nn.Conv2d(in_channels=in_channels, out_channels=ch, kernel_size=3, stride=1, padding=1)
        curr_resolution = resolution

        self.down = nn.ModuleList()
        in_ch_mult = (1,) + tuple(ch_mult)

        for i_level in range(self.num_resolutions):
          block = nn.ModuleList()
          attn = nn.ModuleList()

          block_in = ch * in_ch_mult[i_level]
          block_out = ch * ch_mult[i_level]

          for i_block in range(self.num_res_blocks):
            block.append(ResnetBlock(in_channels=block_in, out_channels=block_out, dropout=dropout))

            block_in = block_out

            if curr_resolution in attn_resolutions:
              attn.append(AttnBlock(block_in))

          down = nn.Module()
          down.block = block
          down.attn = attn

          # do not downsample at the final resolution
          if i_level != self.num_resolutions - 1:
            down.downsample = Downsample(block_in, resamp_with_conv)
            curr_resolution = curr_resolution // 2

          self.down.append(down)

        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)
        self.mid.attn_1 = AttnBlock(block_in)

        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)

        self.norm_out = Normalize(block_in)
        self.conv_out = nn.Conv2d(block_in, z_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
      h = self.conv_in(x)                             # 256x256x3
      for i_level in range(self.num_resolutions):
        for i_block in range(self.num_res_blocks):
          h = self.down[i_level].block[i_block](h)  # 256x256x128 -> 128x128x128 -> 64x64x128 -> 32x32x256 -> 16x16x512
          if len(self.down[i_level].attn) > 0:
            h = self.down[i_level].attn[i_block](h)

        if i_level != self.num_resolutions - 1:
            h = self.down[i_level].downsample(h)

      h = self.mid.block_1(h)
      h = self.mid.attn_1(h)
      h = self.mid.block_2(h)

      h = self.norm_out(h)
      h = nonlinearity(h)
      h = self.conv_out(h)                            # 16x16x256
      return h

In [ ]:
class Decoder(nn.Module):
    def __init__(self, out_channels=3, ch=128, ch_mult=(1, 1, 2, 2, 4), num_res_blocks=2, attn_resolutions=(16,), resolution=256, z_channels=256, dropout=0.0, resamp_with_conv=True):
        super().__init__()
        self.ch = ch
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution

        block_in = ch * ch_mult[-1]
        curr_resolution = resolution // (2 ** (self.num_resolutions -1))

        self.conv_in = nn.Conv2d(z_channels, block_in, kernel_size=3, stride=1, padding=1)
        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)
        self.mid.attn_1 = AttnBlock(block_in)
        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)

        self.up = nn.ModuleList()

        for i_level in reversed(range(self.num_resolutions)):
          block = nn.ModuleList()
          attn = nn.ModuleList()

          block_out = ch * ch_mult[i_level]

          for i_block in range(num_res_blocks + 1):
            block.append(ResnetBlock(in_channels=block_in, out_channels=block_out, dropout=dropout))

            block_in = block_out

            if curr_resolution in attn_resolutions:
              attn.append(AttnBlock(block_in))

          up = nn.Module()
          up.block = block
          up.attn = attn

          # upsample except at final stage
          if i_level != 0:
            up.upsample = Upsample(block_in, resamp_with_conv)
            curr_resolution = curr_resolution * 2

          self.up.append(up)

        self.norm_out = Normalize(block_in)
        self.conv_out = nn.Conv2d(block_in, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
      h = self.conv_in(x)

      h = self.mid.block_1(h)
      h = self.mid.attn_1(h)
      h = self.mid.block_2(h)

      for i_level in range(self.num_resolutions):
        for i_block in range(self.num_res_blocks + 1):
          h = self.up[i_level].block[i_block](h)
          if len(self.up[i_level].attn) > 0:
            h = self.up[i_level].attn[i_block](h)
        if i_level != self.num_resolutions - 1:
          h = self.up[i_level].upsample(h)

      h = self.norm_out(h)
      h = nonlinearity(h)
      h = self.conv_out(h)
      return h

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_permuted = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_permuted.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        active_codes = torch.unique(encoding_indices).numel()

        return quantized, loss, perplexity, active_codes

In [ ]:
class VQGAN(nn.Module):
    def __init__(self, resolution=256, in_channels=3, out_channels=3, ch=128, ch_mult=(1, 1, 2, 2, 4), num_res_blocks=2, attn_resolutions=(16,), z_channels=256, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25, dropout=0.0, resamp_with_conv=True):
        super().__init__()
        self.encoder = Encoder(in_channels, ch, ch_mult, num_res_blocks, attn_resolutions, resolution, z_channels, dropout, resamp_with_conv)
        self.quant_conv = nn.Conv2d(z_channels, embedding_dim,kernel_size=1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.post_quant_conv = nn.Conv2d(embedding_dim, z_channels,kernel_size=1)
        self.decoder = Decoder(out_channels, ch, ch_mult, num_res_blocks, attn_resolutions, resolution, z_channels, dropout, resamp_with_conv)

    def forward(self, x):
        z = self.encoder(x)
        z = self.quant_conv(z)

        quantized, vq_loss, perplexity, active_codes = self.vq(z)
        quantized = self.post_quant_conv(quantized)

        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity, active_codes

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim, kernel_size=4, stride=2, padding=1)                   # (3, 256, 256) -> (64, 128, 128)
        self.relu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # (64, 128, 128) -> (128, 64, 64)
        self.norm1 = nn.BatchNorm2d(hidden_dim * 2)

        self.conv3 = nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)  # (128, 64, 64) -> (256, 32, 32)
        self.norm2 = nn.BatchNorm2d(hidden_dim * 4)

        self.conv4 = nn.Conv2d(hidden_dim * 4, 1, kernel_size=4, stride=1, padding=1)               # (256, 32, 32) -> (1, 31, 31)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.norm1(self.conv2(x)))
        x = self.relu(self.norm2(self.conv3(x)))
        x = self.conv4(x)
        return x